# 🔧 Function Calling Basics

**Day 10 — Notebook 1 of 5 | Estimated Time: 30 minutes**

---

## What You'll Learn
- What function calling is and why it matters
- How Gemini decides when to call a function vs answer directly
- Declaring functions with type-safe schemas
- The full function-calling request/response cycle
- Handling parallel and sequential function calls

---

## What Is Function Calling?

Without function calling:
```
User: "What's 847 × 293?"
LLM: "It's approximately 248,171"  ← may be wrong, no real computation
```

With function calling:
```
User: "What's 847 × 293?"
LLM: → calls multiply(847, 293) → gets 248,171
LLM: "847 × 293 = 248,171"  ← computed, not guessed
```

Function calling lets the LLM **delegate** tasks it can't do reliably itself  
(real-time data, precise computation, database queries, API calls).

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os, json

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
MODEL = "gemini-2.5-flash"

---

## 1. Declaring a Function

A function declaration tells Gemini:
- What the function is called
- What it does (description — the LLM reads this to decide when to call it)
- What parameters it accepts (names, types, descriptions)

In [ ]:
# Declare a simple calculator tool
calculator_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="calculate",
            description="Perform basic arithmetic: add, subtract, multiply, or divide two numbers.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "operation": types.Schema(
                        type=types.Type.STRING,
                        description="The arithmetic operation: 'add', 'subtract', 'multiply', or 'divide'",
                        enum=["add", "subtract", "multiply", "divide"],
                    ),
                    "a": types.Schema(
                        type=types.Type.NUMBER,
                        description="The first number",
                    ),
                    "b": types.Schema(
                        type=types.Type.NUMBER,
                        description="The second number",
                    ),
                },
                required=["operation", "a", "b"],
            ),
        )
    ]
)

# The actual Python function that executes the operation
def calculate(operation: str, a: float, b: float) -> dict:
    """Execute the requested arithmetic operation."""
    ops = {
        "add":      lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide":   lambda x, y: x / y if y != 0 else None,
    }
    result = ops[operation](a, b)
    if result is None:
        return {"error": "Division by zero"}
    return {"result": result}


print("Calculator tool declared.")
print(f"Parameters: {[p for p in ['operation', 'a', 'b']]}")

---

## 2. The Function-Calling Cycle

```
Step 1: Send query + tool declarations to Gemini
Step 2: Gemini returns a FunctionCall (name + args) — not the final answer
Step 3: Your code executes the function
Step 4: Send function result back to Gemini
Step 5: Gemini returns the final natural language answer
```

In [ ]:
def run_with_tools(query: str, tools: list, verbose: bool = True) -> str:
    """
    Execute a query with function calling — handles a single function call.
    Returns the final natural language answer.
    """
    TOOL_REGISTRY = {"calculate": calculate}

    # Step 1: Send query
    response = client.models.generate_content(
        model=MODEL,
        contents=query,
        config=types.GenerateContentConfig(tools=tools),
    )

    # Step 2: Check if Gemini wants to call a function
    part = response.candidates[0].content.parts[0]

    if not hasattr(part, "function_call") or part.function_call is None:
        # No function call — Gemini answered directly
        if verbose:
            print("  → Gemini answered directly (no function call)")
        return part.text

    fn_call = part.function_call
    fn_name = fn_call.name
    fn_args = dict(fn_call.args)

    if verbose:
        print(f"  → Gemini called: {fn_name}({fn_args})")

    # Step 3: Execute the function
    if fn_name not in TOOL_REGISTRY:
        fn_result = {"error": f"Unknown function: {fn_name}"}
    else:
        fn_result = TOOL_REGISTRY[fn_name](**fn_args)

    if verbose:
        print(f"  → Function returned: {fn_result}")

    # Step 4: Send result back to Gemini
    follow_up = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Content(role="user", parts=[types.Part(text=query)]),
            types.Content(
                role="model",
                parts=[types.Part(function_call=fn_call)],
            ),
            types.Content(
                role="user",
                parts=[
                    types.Part(
                        function_response=types.FunctionResponse(
                            name=fn_name,
                            response=fn_result,
                        )
                    )
                ],
            ),
        ],
        config=types.GenerateContentConfig(tools=tools),
    )

    # Step 5: Return final answer
    return follow_up.text.strip()


# Test it
queries = [
    "What is 847 multiplied by 293?",
    "Divide 1000 by 7 and tell me the result.",
    "What is the capital of France?",  # No function call needed
]

for q in queries:
    print(f"\nQuery: {q}")
    answer = run_with_tools(q, [calculator_tool])
    print(f"Answer: {answer}")

---

## 3. Multiple Tools

Gemini can choose between multiple tools and picks the right one based on the descriptions.

In [ ]:
# Unit converter tool
def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    """Convert between common units."""
    conversions = {
        ("km", "miles"): 0.621371,
        ("miles", "km"): 1.60934,
        ("kg", "lbs"): 2.20462,
        ("lbs", "kg"): 0.453592,
        ("celsius", "fahrenheit"): None,  # special case
        ("fahrenheit", "celsius"): None,
        ("litres", "gallons"): 0.264172,
        ("gallons", "litres"): 3.78541,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key == ("celsius", "fahrenheit"):
        return {"result": value * 9/5 + 32, "unit": "°F"}
    if key == ("fahrenheit", "celsius"):
        return {"result": (value - 32) * 5/9, "unit": "°C"}
    if key not in conversions:
        return {"error": f"Cannot convert {from_unit} to {to_unit}"}
    return {"result": round(value * conversions[key], 4), "unit": to_unit}


converter_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="convert_units",
            description="Convert a value between units of measurement such as km/miles, kg/lbs, celsius/fahrenheit, litres/gallons.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "value":     types.Schema(type=types.Type.NUMBER, description="The numeric value to convert"),
                    "from_unit": types.Schema(type=types.Type.STRING, description="The source unit (e.g. 'km', 'celsius', 'kg')"),
                    "to_unit":   types.Schema(type=types.Type.STRING, description="The target unit (e.g. 'miles', 'fahrenheit', 'lbs')"),
                },
                required=["value", "from_unit", "to_unit"],
            ),
        )
    ]
)

TOOL_REGISTRY = {"calculate": calculate, "convert_units": convert_units}
all_tools = [calculator_tool, converter_tool]


def run_with_multi_tools(query: str, verbose: bool = True) -> str:
    """Same cycle as run_with_tools but uses the full TOOL_REGISTRY."""
    response = client.models.generate_content(
        model=MODEL, contents=query,
        config=types.GenerateContentConfig(tools=all_tools),
    )
    part = response.candidates[0].content.parts[0]

    if not hasattr(part, "function_call") or part.function_call is None:
        return part.text

    fn_call = part.function_call
    fn_name, fn_args = fn_call.name, dict(fn_call.args)

    if verbose:
        print(f"  → Called: {fn_name}({fn_args})")

    fn_result = TOOL_REGISTRY[fn_name](**fn_args) if fn_name in TOOL_REGISTRY else {"error": "unknown"}

    if verbose:
        print(f"  → Result: {fn_result}")

    follow_up = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Content(role="user", parts=[types.Part(text=query)]),
            types.Content(role="model", parts=[types.Part(function_call=fn_call)]),
            types.Content(role="user", parts=[types.Part(
                function_response=types.FunctionResponse(name=fn_name, response=fn_result)
            )]),
        ],
        config=types.GenerateContentConfig(tools=all_tools),
    )
    return follow_up.text.strip()


multi_queries = [
    "How many miles is 42.195 km?",
    "What is 15% of 340?",
    "Convert 100 Celsius to Fahrenheit.",
]

for q in multi_queries:
    print(f"\nQuery: {q}")
    print(f"Answer: {run_with_multi_tools(q)}")

---

## 4. Tool Choice Control

You can control when Gemini uses tools with `tool_config`.

In [ ]:
query = "What is 50 divided by 8?"

# AUTO (default) — Gemini decides whether to call a function
resp_auto = client.models.generate_content(
    model=MODEL, contents=query,
    config=types.GenerateContentConfig(
        tools=all_tools,
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode=types.FunctionCallingConfigMode.AUTO
            )
        ),
    ),
)

# NONE — force Gemini to answer without using tools
resp_none = client.models.generate_content(
    model=MODEL, contents=query,
    config=types.GenerateContentConfig(
        tools=all_tools,
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode=types.FunctionCallingConfigMode.NONE
            )
        ),
    ),
)

part_auto = resp_auto.candidates[0].content.parts[0]
part_none = resp_none.candidates[0].content.parts[0]

print(f"Query: '{query}'\n")
auto_result = "called function" if hasattr(part_auto, "function_call") and part_auto.function_call else part_auto.text[:80]
print(f"  AUTO mode: {auto_result}")
print(f"  NONE mode: {part_none.text[:80]}")  # must answer directly

print("""
Tool config modes:
  AUTO  — Gemini decides (default, recommended)
  ANY   — Gemini must call at least one tool
  NONE  — Gemini cannot call tools (forces direct answer)
""")

---

## 5. Structured Output via Function Calling

A useful pattern: declare a "fake" function that Gemini is forced to call  
to extract structured data from unstructured text.

In [ ]:
# Extract structured entity data from free text
extract_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="extract_event",
            description="Extract structured event details from text.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "event_name": types.Schema(type=types.Type.STRING, description="Name of the event"),
                    "date":       types.Schema(type=types.Type.STRING, description="Date of the event (ISO format if possible)"),
                    "location":   types.Schema(type=types.Type.STRING, description="Location or venue"),
                    "organiser":  types.Schema(type=types.Type.STRING, description="Person or organisation running the event"),
                    "cost_gbp":   types.Schema(type=types.Type.NUMBER, description="Cost in GBP, or 0 if free"),
                },
                required=["event_name", "date", "location"],
            ),
        )
    ]
)

event_texts = [
    "Join us for the London AI Summit on March 15th at the Barbican Centre. "
    "Organised by TechHub London, tickets are £120 per person.",

    "Free Python workshop this Saturday at CodeSpace Manchester. "
    "Hosted by Manchester PyLadies, all skill levels welcome.",
]

print("Extracting structured event data:\n")
for text in event_texts:
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Extract event details from: {text}",
        config=types.GenerateContentConfig(
            tools=[extract_tool],
            tool_config=types.ToolConfig(
                function_calling_config=types.FunctionCallingConfig(
                    mode=types.FunctionCallingConfigMode.ANY,  # force a call
                    allowed_function_names=["extract_event"],
                )
            ),
        ),
    )
    part = response.candidates[0].content.parts[0]
    extracted = dict(part.function_call.args)
    print(f"Input:  {text[:60]}...")
    print(f"Output: {json.dumps(extracted, indent=2)}")
    print()

---

## 🏋️ Exercise 1: Add a String Tool

Build a tool that handles string operations the LLM might get wrong.

In [ ]:
# Exercise 1: String tools
# LLMs are notoriously bad at: counting characters, reversing strings,
# and exact letter counting (e.g., "how many r's in 'strawberry'?")
#
# TODO:
# 1. Implement string_tool() function that supports operations:
#    - "count_chars"   → count total characters
#    - "count_letter" → count occurrences of a specific letter
#    - "reverse"      → reverse the string
#    - "uppercase" / "lowercase"
#
# 2. Create the types.Tool declaration with correct schema
#
# 3. Add to TOOL_REGISTRY and test on:
#    - "How many letters are in the word 'extraordinary'?"
#    - "How many r's are in 'strawberry'?"
#    - "Reverse the word 'racecar'."

def string_tool(operation: str, text: str, letter: str = "") -> dict:
    # TODO: Implement
    pass

string_tool_decl = None  # TODO: Create types.Tool

# TODO: Test your tool

---

## 🏋️ Exercise 2: Multi-Step Calculation

Test whether Gemini can chain multiple function calls to solve a compound problem.

In [ ]:
# Exercise 2: Multi-step problem
# Some problems require multiple function calls in sequence.
# For example: "I ran 5km. How many miles is that? And if I ran at 12km/h, how long did it take in minutes?"
#
# Extend run_with_multi_tools() to support a loop:
#   - Keep calling functions until Gemini returns text (no more function calls)
#   - Print each step: which function was called, what it returned
#   - Cap at 10 iterations to prevent infinite loops
#
# Test on: "If a marathon is 42.195 km, and I ran it in 3h 45min,
#            what was my average speed in km/h and in miles per hour?"

def run_agentic_loop(query: str, max_steps: int = 10) -> str:
    """Handle multi-turn function calling until Gemini returns a final answer."""
    contents = [types.Content(role="user", parts=[types.Part(text=query)])]
    step = 0

    while step < max_steps:
        step += 1
        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config=types.GenerateContentConfig(tools=all_tools),
        )
        part = response.candidates[0].content.parts[0]

        # No function call → final answer
        if not hasattr(part, "function_call") or part.function_call is None:
            return part.text.strip()

        # TODO: Execute function, append result to contents, continue loop
        fn_call = part.function_call
        fn_name, fn_args = fn_call.name, dict(fn_call.args)
        print(f"  Step {step}: {fn_name}({fn_args})")

        fn_result = TOOL_REGISTRY.get(fn_name, lambda **k: {"error": "unknown"})(**fn_args)
        print(f"           → {fn_result}")

        # Append model's function call and the result
        contents.append(types.Content(role="model", parts=[types.Part(function_call=fn_call)]))
        contents.append(types.Content(role="user", parts=[types.Part(
            function_response=types.FunctionResponse(name=fn_name, response=fn_result)
        )]))

    return "Reached maximum steps without a final answer."


print("Multi-step test:")
answer = run_agentic_loop(
    "A marathon is 42.195 km. If I finished in 3 hours 45 minutes, "
    "what was my average speed in km/h? Then convert that to miles per hour."
)
print(f"\nFinal answer: {answer}")

---

## Key Takeaways

| Concept | Detail |
|---------|--------|
| Tool declaration | `types.FunctionDeclaration` with name, description, and typed schema |
| Decision making | Gemini reads the **description** to decide when to call a function |
| Cycle | Query → FunctionCall → execute → FunctionResponse → final answer |
| Tool config | `AUTO` (default) / `ANY` (force call) / `NONE` (no tools) |
| Structured extraction | Use `ANY` mode + a "fake" extraction function for reliable JSON output |
| Agentic loop | Repeat until no function call → Gemini gives text answer |

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [Gemini Function Calling](https://ai.google.dev/gemini-api/docs/function-calling) | Docs | Official reference with more examples |
| [Tool Use Best Practices](https://ai.google.dev/gemini-api/docs/function-calling#best-practices) | Docs | Tips for writing good tool descriptions |

---

**Next up:** [02_Building_Custom_Tools.ipynb](./02_Building_Custom_Tools.ipynb)